<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Gemini API and VertexAI</h1>
<h1>Multimodal Applications</h1>
        <p>Bruno Gon&#231;alves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [ ]:
from collections import Counter
from pprint import pprint
import os
import requests
from io import BytesIO
from PIL import Image
import base64

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import google.generativeai as genai
import vertexai
from vertexai.generative_models import GenerativeModel, Part, Image as VertexImage

import watermark
%load_ext watermark
%matplotlib inline

We start by printing out the versions of the libraries we're using for future reference

In [ ]:
%watermark -n -v -m -g -iv

Load default figure style

In [ ]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# Section 1: Gemini's Image and Video Understanding Capabilities

Gemini is natively multimodal, meaning it was trained on text, images, audio, and video simultaneously. This gives it a fundamentally different approach to understanding media compared to models that use separate encoders.

### Supported Modalities

| Modality  | Supported Formats                              |
|-----------|------------------------------------------------|
| Images    | JPEG, PNG, WEBP, HEIC, HEIF                    |
| Video     | MP4, MPEG, MOV, AVI, FLV, MKV, WEBM           |
| Audio     | WAV, MP3, AIFF, AAC, OGG, FLAC                |
| Documents | PDF, plain text                                |

## 1.1 Setup: Configure Gemini API

We configure the Gemini SDK using an API key. In production environments you would use Vertex AI with service account credentials instead.

In [ ]:
# Configure the Gemini API with your API key.
# Store the key in an environment variable rather than hard-coding it.
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "YOUR_API_KEY_HERE")
genai.configure(api_key=GOOGLE_API_KEY)

# Choose the Gemini model. gemini-1.5-flash is fast and cost-effective;
# gemini-1.5-pro offers higher quality for complex tasks.
MODEL_NAME = "gemini-1.5-flash"
model = genai.GenerativeModel(MODEL_NAME)

print(f"Model configured: {MODEL_NAME}")

## 1.2 Loading and Displaying an Image

We download a publicly available image and display it in the notebook before passing it to Gemini.

In [ ]:
# Download a well-known public image from Wikipedia (Creative Commons licensed).
# This is a photo of the Golden Gate Bridge.
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/0/0c/GoldenGateBridge-001.jpg/1280px-GoldenGateBridge-001.jpg"

def load_image_from_url(url):
    """Download an image from a URL and return a PIL Image object."""
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    return Image.open(BytesIO(response.content))

sample_image = load_image_from_url(IMAGE_URL)

# Display the image inline
plt.figure(figsize=(10, 6))
plt.imshow(sample_image)
plt.axis('off')
plt.title("Sample image: Golden Gate Bridge (Wikimedia Commons)", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Image size: {sample_image.size}  |  Mode: {sample_image.mode}")

## 1.3 Describing an Image with Gemini

The simplest multimodal call: pass the image alongside a text prompt.

In [ ]:
# Pass a PIL Image directly to generate_content.
# The SDK automatically converts it to the required inline bytes format.
response = model.generate_content([
    "Describe this image in detail. Include the main subject, setting, colors, and any interesting details.",
    sample_image
])

print("=== Gemini's Description ===")
print(response.text)

## 1.4 Passing Images in Different Formats

Gemini accepts images as PIL objects, raw bytes, or base64-encoded strings. Understanding the differences helps you choose the right approach for your architecture.

In [ ]:
# --- Method 1: PIL Image (most convenient for notebook workflows) ---
response_pil = model.generate_content(["What is shown in this image?", sample_image])
print("Method 1 - PIL Image:")
print(response_pil.text[:200], "...\n")

# --- Method 2: Raw bytes (useful when reading from disk or network) ---
buffer = BytesIO()
sample_image.save(buffer, format="JPEG")
image_bytes = buffer.getvalue()

response_bytes = model.generate_content([
    "What is shown in this image?",
    {"mime_type": "image/jpeg", "data": image_bytes}
])
print("Method 2 - Raw bytes:")
print(response_bytes.text[:200], "...\n")

# --- Method 3: Base64-encoded string (common in REST API workflows) ---
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

response_b64 = model.generate_content([
    "What is shown in this image?",
    {"mime_type": "image/jpeg", "data": base64.b64decode(image_b64)}
])
print("Method 3 - Base64:")
print(response_b64.text[:200], "...")

## 1.5 Video Understanding

Gemini can analyze video content. For short videos you can upload them with the File API; for publicly hosted videos you can reference them via URI. Below we demonstrate the File API approach with a small public video.

In [ ]:
# NOTE: Video understanding via the File API requires uploading the file first.
# The code below shows the complete pattern. To run it you need a valid API key
# with File API access and a video file path.

def describe_video(video_path: str, prompt: str = "Describe what happens in this video.") -> str:
    """
    Upload a local video file to the Gemini File API and request a description.

    Args:
        video_path: Local path to the video file.
        prompt: Text prompt to accompany the video.

    Returns:
        Model response text.
    """
    import time

    print(f"Uploading {video_path} ...")
    video_file = genai.upload_file(path=video_path)

    # The File API processes the video asynchronously; wait until it is ACTIVE.
    while video_file.state.name == "PROCESSING":
        print("Processing video...", end="\r")
        time.sleep(5)
        video_file = genai.get_file(video_file.name)

    if video_file.state.name == "FAILED":
        raise ValueError(f"Video processing failed: {video_file.state}")

    print(f"Video ready: {video_file.uri}")
    response = model.generate_content([prompt, video_file])
    return response.text


# Example usage (commented out – requires a local video file):
# result = describe_video("my_video.mp4", "Summarize the key events in this video.")
# print(result)

print("describe_video() function is defined and ready to use.")
print("Supported formats: MP4, MPEG, MOV, AVI, FLV, MKV, WEBM")

# Section 2: Image Captioning and Visual Question Answering (VQA)

Image captioning and VQA are two of the most common multimodal tasks:

- **Captioning**: produce a natural language description of an image (useful for accessibility, content indexing, SEO).
- **VQA**: answer specific questions about the content of an image (useful for information extraction, customer support, e-commerce).

Gemini handles both with the same API — the prompt style is what differentiates them.

## 2.1 Building an Image Dataset

We collect several publicly available images covering different domains to demonstrate the breadth of Gemini's vision capabilities.

In [ ]:
# A small collection of Creative Commons / public domain images from Wikimedia.
image_urls = {
    "golden_gate": "https://upload.wikimedia.org/wikipedia/commons/thumb/0/0c/GoldenGateBridge-001.jpg/640px-GoldenGateBridge-001.jpg",
    "eiffel_tower": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/85/Smiley.svg/200px-Smiley.svg.png",
    "cat": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/640px-Cat_November_2010-1a.jpg",
    "market": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/640px-Cat03.jpg",
}

# Download all images and store as PIL objects
images = {}
for name, url in image_urls.items():
    try:
        images[name] = load_image_from_url(url)
        print(f"  Loaded '{name}': {images[name].size}")
    except Exception as e:
        print(f"  Could not load '{name}': {e}")

print(f"\nLoaded {len(images)} images.")

In [ ]:
# Display the loaded images in a grid
n = len(images)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))

if n == 1:
    axes = [axes]

for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img)
    ax.set_title(name.replace("_", " ").title(), fontsize=11)
    ax.axis('off')

plt.suptitle("Image Dataset", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2.2 Image Captioning

Captioning prompts should ask for a specific style: short (tweet-like), medium (one-sentence), or long (paragraph). The right level of detail depends on your use case.

In [ ]:
def caption_image(image: Image.Image, style: str = "detailed") -> str:
    """
    Generate a caption for the given image.

    Args:
        image: PIL Image to caption.
        style: 'short', 'medium', or 'detailed'.

    Returns:
        Caption string.
    """
    prompts = {
        "short": "Write a short, engaging caption for this image in one sentence (max 15 words).",
        "medium": "Write a one-sentence caption that accurately describes the main subject and setting of this image.",
        "detailed": (
            "Write a detailed caption for this image that covers: "
            "(1) the main subject, (2) the setting or background, "
            "(3) notable colors or lighting, and (4) the overall mood."
        ),
    }
    prompt = prompts.get(style, prompts["medium"])
    response = model.generate_content([prompt, image])
    return response.text.strip()


# Generate captions at all three levels for the first image
demo_image = list(images.values())[0]
demo_name = list(images.keys())[0]

print(f"Image: {demo_name}\n")
for style in ["short", "medium", "detailed"]:
    caption = caption_image(demo_image, style=style)
    print(f"[{style.upper()}]\n{caption}\n")

## 2.3 Detailed Image Analysis: Objects, Colors, and Text Detection

In [ ]:
def analyze_image(image: Image.Image) -> dict:
    """
    Ask Gemini to perform a structured analysis of the image.
    Returns a dictionary with objects, colors, text, and overall description.
    """
    prompt = """\
Analyze this image and provide a structured response with the following sections:

OBJECTS: List the main objects or subjects visible (comma-separated).
COLORS: List the dominant colors (comma-separated).
TEXT: Any readable text visible in the image (or 'None' if absent).
SCENE: One sentence describing the overall scene.
MOOD: One word describing the mood or atmosphere.

Use exactly these section headers followed by a colon."""

    response = model.generate_content([prompt, image])
    raw = response.text

    # Parse the structured output into a dictionary
    result = {}
    for line in raw.splitlines():
        if ":" in line:
            key, _, value = line.partition(":")
            key = key.strip().lower()
            if key in {"objects", "colors", "text", "scene", "mood"}:
                result[key] = value.strip()
    return result


analysis = analyze_image(demo_image)
print(f"Analysis of '{demo_name}':")
pprint(analysis)

## 2.4 Visual Question Answering (VQA)

VQA lets users ask arbitrary natural-language questions about an image. Unlike captioning, VQA is interactive and targeted — the answer depends entirely on what you ask.

In [ ]:
def vqa(image: Image.Image, question: str) -> str:
    """Answer a question about the given image."""
    prompt = f"Question about the image: {question}\nProvide a concise, factual answer."
    response = model.generate_content([prompt, image])
    return response.text.strip()


# Ask a set of questions about the demo image
questions = [
    "What is the main subject of this image?",
    "What time of day does this appear to be?",
    "What colors are most prominent?",
    "Is there water visible in this image?",
    "How would you describe the weather conditions?",
]

print(f"VQA on '{demo_name}':\n")
for q in questions:
    answer = vqa(demo_image, q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

## 2.5 Comparing Responses with Different Prompt Styles

The specificity and framing of a prompt dramatically affect the output. Here we compare three approaches for the same question.

In [ ]:
# Demonstrate how prompt engineering changes the output
topic = "the structure visible in this image"

prompt_styles = {
    "Open-ended": f"Tell me about {topic}.",
    "Specific": f"What type of structure is shown? Describe its architectural features.",
    "Constrained": (
        f"In exactly three bullet points, describe {topic}. "
        "Each bullet should be under 20 words."
    ),
}

for style_name, prompt in prompt_styles.items():
    response = model.generate_content([prompt, demo_image])
    print(f"--- {style_name} ---")
    print(response.text.strip())
    print()

## 2.6 Batch Processing: Captions and Analysis for All Images

We process the entire image dataset in a loop and collect results in a Pandas DataFrame for easy exploration.

In [ ]:
records = []

for name, img in images.items():
    print(f"Processing '{name}'...")
    try:
        short_caption = caption_image(img, style="short")
        analysis = analyze_image(img)
        records.append({
            "image": name,
            "short_caption": short_caption,
            "objects": analysis.get("objects", ""),
            "colors": analysis.get("colors", ""),
            "mood": analysis.get("mood", ""),
            "scene": analysis.get("scene", ""),
        })
    except Exception as e:
        print(f"  Error: {e}")

df = pd.DataFrame(records)
print("\nBatch results:")
df

In [ ]:
# Visualize mood distribution across images
if not df.empty and "mood" in df.columns:
    mood_counts = Counter(df["mood"].str.lower().tolist())

    fig, ax = plt.subplots(figsize=(8, 4))
    labels, values = zip(*mood_counts.most_common()) if mood_counts else ([], [])
    ax.barh(labels, values)
    ax.set_xlabel("Count")
    ax.set_title("Mood Distribution Across Images")
    plt.tight_layout()
    plt.show()

# Section 3: Integrating Multimodal Inputs

The real power of Gemini emerges when you combine multiple modalities and inputs in a single prompt. This section demonstrates how to:

- Mix text and images in one call
- Send multiple images simultaneously
- Use structured prompts for reliable output formats
- Assemble multimodal payloads using the Vertex AI `Part` API

## 3.1 Text + Image: Document and Chart Analysis

In [ ]:
# Generate a sample chart programmatically and ask Gemini to interpret it.
# This simulates the common pattern of analyzing charts from reports or dashboards.

fig, ax = plt.subplots(figsize=(8, 5))
quarters = ["Q1", "Q2", "Q3", "Q4"]
revenue  = [120, 145, 132, 178]   # fictional revenue in $M
expenses = [ 95, 110, 108, 130]   # fictional expenses in $M

x = np.arange(len(quarters))
width = 0.35
ax.bar(x - width/2, revenue,  width, label="Revenue ($M)")
ax.bar(x + width/2, expenses, width, label="Expenses ($M)")
ax.set_xticks(x)
ax.set_xticklabels(quarters)
ax.set_ylabel("Amount ($M)")
ax.set_title("Acme Corp – Quarterly Financial Summary")
ax.legend()
plt.tight_layout()

# Save the chart to a buffer and load as PIL Image
buf = BytesIO()
plt.savefig(buf, format="png", dpi=120)
buf.seek(0)
chart_image = Image.open(buf).copy()
plt.show()

# Ask Gemini to interpret the chart
chart_prompt = """\
This is a bar chart from a financial report. Please:
1. Identify the key trends shown.
2. Highlight the most notable quarter.
3. Calculate the approximate profit margin for Q4.
4. Provide a one-sentence executive summary."""

chart_response = model.generate_content([chart_prompt, chart_image])
print("=== Chart Analysis ===")
print(chart_response.text)

## 3.2 Multiple Images in a Single Prompt

Gemini can reason across multiple images at once — useful for comparison, change detection, or multi-perspective analysis.

In [ ]:
# Use the first two images from our dataset for a comparison
img_names = list(images.keys())[:2]
img_list  = [images[n] for n in img_names]

if len(img_list) >= 2:
    multi_prompt = (
        f"I am showing you two images. "
        f"Image 1: {img_names[0].replace('_', ' ')}. "
        f"Image 2: {img_names[1].replace('_', ' ')}. "
        "Please compare them by: "
        "(1) listing what they have in common, "
        "(2) describing the key differences, and "
        "(3) identifying which image would be more suitable for a travel advertisement and why."
    )

    # The Gemini SDK accepts a flat list: [prompt_text, img1, img2, ...]
    multi_response = model.generate_content([multi_prompt] + img_list)
    print("=== Multi-Image Comparison ===")
    print(multi_response.text)
else:
    print("Need at least two images loaded to run this cell.")

## 3.3 Structured Prompts for Product Catalog Analysis

Structured prompts make Gemini output parseable JSON or CSV that can feed directly into databases or downstream pipelines.

In [ ]:
import json

def extract_product_info(image: Image.Image) -> dict:
    """
    Treat the image as a product photo and extract structured catalog data.
    Returns a dictionary with product attributes.
    """
    prompt = """\
Imagine this image is a product photo for an online store.
Extract the following attributes and respond ONLY with valid JSON — no extra text:
{
  "product_name": "<short descriptive name>",
  "category": "<electronics|clothing|outdoor|food|other>",
  "primary_color": "<color>",
  "key_features": ["<feature1>", "<feature2>", "<feature3>"],
  "suggested_tags": ["<tag1>", "<tag2>", "<tag3>"],
  "alt_text": "<accessible image description in one sentence>"
}"""

    response = model.generate_content([prompt, image])
    raw = response.text.strip()

    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Fall back to returning the raw text in a dict
        return {"raw_response": raw}


# Apply to the demo image
product_info = extract_product_info(demo_image)
print("=== Product Catalog Entry ===")
pprint(product_info)

## 3.4 Using Vertex AI `Part` Objects for Multimodal Assembly

When using Vertex AI (as opposed to the `google.generativeai` SDK), multimodal inputs are assembled using `Part` objects. This is the production-grade approach for GCP-hosted applications.

In [ ]:
# --- Vertex AI Part-based multimodal assembly ---
# This pattern is used when your application runs on Google Cloud Platform.
#
# Uncomment and fill in your project details to run:
#
# PROJECT_ID = "your-gcp-project-id"
# LOCATION   = "us-central1"
#
# vertexai.init(project=PROJECT_ID, location=LOCATION)
# vertex_model = GenerativeModel("gemini-1.5-flash")
#
# # Convert a PIL image to bytes for Vertex AI
# buf = BytesIO()
# demo_image.save(buf, format="JPEG")
# image_bytes = buf.getvalue()
#
# # Build a multimodal content list using Part objects
# contents = [
#     Part.from_text("Describe this image in detail:"),
#     Part.from_image(VertexImage.from_bytes(image_bytes)),
# ]
#
# response = vertex_model.generate_content(contents)
# print(response.text)

print("Vertex AI Part API example shown above (requires GCP project credentials).")
print()
print("Part types available:")
print("  Part.from_text(text)            - Text content")
print("  Part.from_image(VertexImage)    - Image content")
print("  Part.from_uri(uri, mime_type)   - Cloud Storage URI")
print("  Part.from_data(data, mime_type) - Raw bytes")

## 3.5 Audio Understanding

Gemini can transcribe, translate, and reason about audio content. The workflow is identical to video: upload via the File API, then include the file reference in your prompt.

In [ ]:
def describe_audio(audio_path: str, prompt: str = "Transcribe this audio and summarize the main points.") -> str:
    """
    Upload an audio file to the Gemini File API and request analysis.

    Supported formats: WAV, MP3, AIFF, AAC, OGG, FLAC.

    Args:
        audio_path: Local path to the audio file.
        prompt: Text prompt describing the task.

    Returns:
        Model response text.
    """
    import time

    audio_file = genai.upload_file(path=audio_path)

    while audio_file.state.name == "PROCESSING":
        time.sleep(3)
        audio_file = genai.get_file(audio_file.name)

    response = model.generate_content([prompt, audio_file])
    return response.text


# Example usage (requires a local audio file):
# transcript = describe_audio("meeting_recording.mp3",
#                             "Transcribe this meeting and extract action items.")
# print(transcript)

print("describe_audio() is defined.")
print("Supported audio formats: WAV, MP3, AIFF, AAC, OGG, FLAC")

# Section 4: Real-world Use Cases

This section implements four production-oriented multimodal pipelines that demonstrate how to build complete, end-to-end solutions with Gemini.

| Use Case              | Task                                              |
|-----------------------|---------------------------------------------------|
| Content Moderation    | Detect inappropriate or unsafe content            |
| Accessibility         | Generate alt text for screen readers              |
| Document Processing   | Extract structured data from document images      |
| Visual Search         | Describe and categorize products for search       |

## 4.1 Use Case 1: Content Moderation

Detect whether an image contains inappropriate, unsafe, or policy-violating content. The output is a structured moderation report that can be passed to a human review queue.

In [ ]:
def moderate_image(image: Image.Image) -> dict:
    """
    Perform content moderation on an image.

    Returns a dict with:
        - safe (bool): Whether the image is considered safe.
        - confidence (str): high / medium / low.
        - flags (list): List of detected issues (empty if safe).
        - reason (str): Brief explanation.
    """
    prompt = """\
You are a content moderation system. Analyze this image for policy violations.
Check for: violence, explicit content, hate symbols, dangerous activities, 
personally identifiable information (PII), spam, or misleading content.

Respond ONLY with valid JSON:
{
  "safe": true,
  "confidence": "high",
  "flags": [],
  "reason": "<brief explanation>"
}"""

    response = model.generate_content([prompt, image])
    raw = response.text.strip()

    # Strip markdown fences
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"safe": None, "confidence": "unknown", "flags": [], "reason": raw}


# Run moderation on all loaded images
print("=== Content Moderation Results ===\n")
mod_records = []
for name, img in images.items():
    result = moderate_image(img)
    mod_records.append({"image": name, **result})
    status = "SAFE" if result.get("safe") else "FLAGGED"
    print(f"[{status}] {name}: {result.get('reason', '')}")

mod_df = pd.DataFrame(mod_records)
print("\nModeration DataFrame:")
mod_df[["image", "safe", "confidence", "flags"]]

## 4.2 Use Case 2: Accessibility — Generating Alt Text

Alt text (alternative text) is essential for web accessibility and is required by WCAG guidelines. Gemini can generate high-quality, descriptive alt text that helps visually impaired users understand image content.

In [ ]:
def generate_alt_text(image: Image.Image, context: str = "") -> str:
    """
    Generate WCAG-compliant alt text for an image.

    Args:
        image: PIL Image.
        context: Optional surrounding page context to make alt text more relevant.

    Returns:
        Alt text string (under 125 characters for screen reader compatibility).
    """
    context_line = f"\nPage context: {context}" if context else ""

    prompt = f"""\
Generate concise, descriptive alt text for this image following WCAG 2.1 guidelines.
Rules:
- Be specific and descriptive about the image content.
- Focus on what is meaningful, not decorative details.
- Keep it under 125 characters.
- Do not start with 'Image of' or 'Photo of'.
- Return ONLY the alt text string, nothing else.{context_line}"""

    response = model.generate_content([prompt, image])
    alt = response.text.strip().strip('"')
    return alt


# Generate alt text with and without page context
print("=== Alt Text Generation ===\n")
for name, img in images.items():
    alt_plain   = generate_alt_text(img)
    alt_context = generate_alt_text(img, context="travel blog about iconic landmarks")
    print(f"Image: {name}")
    print(f"  No context : {alt_plain}")
    print(f"  With context: {alt_context}")
    print()

## 4.3 Use Case 3: Document Processing

Extract structured data from images of documents such as invoices, receipts, business cards, or forms. This eliminates manual data entry and enables automated document workflows.

In [ ]:
# Synthesize a fake receipt image for demonstration
fig, ax = plt.subplots(figsize=(4, 6))
ax.axis('off')
receipt_text = """
ACME COFFEE SHOP
123 Main Street, San Francisco CA
Tel: (415) 555-0101
-------------------------------
DATE: 2026-02-25  TIME: 09:32
TABLE: 7   SERVER: Maria
-------------------------------
Cappuccino           $4.75
Blueberry Muffin     $3.50
Avocado Toast        $9.00
Orange Juice         $4.25
-------------------------------
Subtotal:           $21.50
Tax (8.625%):        $1.85
TIP:                 $4.00
-------------------------------
TOTAL:              $27.35
-------------------------------
THANK YOU!
"""
ax.text(0.05, 0.95, receipt_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace')
plt.tight_layout()

buf = BytesIO()
plt.savefig(buf, format="png", dpi=150, bbox_inches='tight',
            facecolor='white')
buf.seek(0)
receipt_image = Image.open(buf).copy()
plt.show()

In [ ]:
def extract_receipt_data(image: Image.Image) -> dict:
    """
    Extract structured data from a receipt image.
    Returns a dictionary with merchant info, line items, and totals.
    """
    prompt = """\
This is an image of a receipt or invoice. Extract all information and return ONLY valid JSON:
{
  "merchant_name": "",
  "merchant_address": "",
  "date": "",
  "time": "",
  "line_items": [
    {"description": "", "amount": 0.00}
  ],
  "subtotal": 0.00,
  "tax": 0.00,
  "tip": 0.00,
  "total": 0.00,
  "currency": "USD"
}"""

    response = model.generate_content([prompt, image])
    raw = response.text.strip()

    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"error": "Failed to parse JSON", "raw": raw}


receipt_data = extract_receipt_data(receipt_image)
print("=== Extracted Receipt Data ===")
pprint(receipt_data)

# Display line items as a DataFrame
if "line_items" in receipt_data:
    items_df = pd.DataFrame(receipt_data["line_items"])
    print("\nLine Items:")
    print(items_df.to_string(index=False))

## 4.4 Use Case 4: Visual Search and Product Categorization

Describe and categorize products from images to enable visual search, automated catalog management, or inventory classification.

In [ ]:
# Taxonomy of product categories for consistent classification
CATEGORY_TAXONOMY = [
    "Electronics", "Clothing & Apparel", "Home & Garden", "Sports & Outdoors",
    "Food & Beverage", "Books & Media", "Toys & Games", "Health & Beauty",
    "Automotive", "Art & Collectibles", "Travel & Landmarks", "Nature & Wildlife",
    "Architecture", "Other"
]

def categorize_for_visual_search(image: Image.Image) -> dict:
    """
    Generate a rich product / content description for visual search indexing.

    Returns structured data including category, search keywords, and SEO description.
    """
    taxonomy_str = ", ".join(CATEGORY_TAXONOMY)

    prompt = f"""\
Analyze this image for a visual search system. Return ONLY valid JSON:
{{
  "title": "<concise, descriptive title under 60 characters>",
  "category": "<one of: {taxonomy_str}>",
  "subcategory": "<specific subcategory>",
  "keywords": ["<kw1>", "<kw2>", "<kw3>", "<kw4>", "<kw5>"],
  "description": "<SEO-friendly description in 1-2 sentences>",
  "dominant_colors": ["<color1>", "<color2>"],
  "style_tags": ["<tag1>", "<tag2>"],
  "similar_searches": ["<query1>", "<query2>"]
}}"""

    response = model.generate_content([prompt, image])
    raw = response.text.strip()

    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"error": "Failed to parse", "raw": raw}


print("=== Visual Search Catalog Entry ===")
vs_result = categorize_for_visual_search(demo_image)
pprint(vs_result)

## 4.5 End-to-End Batch Processing Pipeline

We combine all four use cases into a single pipeline that ingests a list of image URLs and produces a unified structured report.

In [ ]:
def process_image_pipeline(name: str, image: Image.Image) -> dict:
    """
    Run all four use-case analyses on a single image.

    Args:
        name: Identifier for the image.
        image: PIL Image to process.

    Returns:
        Dictionary with results from all stages.
    """
    print(f"  Processing '{name}'...")
    result = {"image": name}

    # Stage 1: Moderation gate — skip downstream processing if flagged
    moderation = moderate_image(image)
    result["moderation"] = moderation

    if not moderation.get("safe", True):
        result["status"] = "FLAGGED"
        result["skip_reason"] = "Failed content moderation"
        return result

    result["status"] = "OK"

    # Stage 2: Accessibility
    result["alt_text"] = generate_alt_text(image)

    # Stage 3: Visual search catalog
    result["catalog"] = categorize_for_visual_search(image)

    # Stage 4: Short caption for display
    result["caption"] = caption_image(image, style="short")

    return result


# Run the pipeline on all images
print("=== Batch Image Processing Pipeline ===\n")
pipeline_results = []

for name, img in images.items():
    try:
        record = process_image_pipeline(name, img)
        pipeline_results.append(record)
    except Exception as e:
        print(f"  ERROR processing '{name}': {e}")
        pipeline_results.append({"image": name, "status": "ERROR", "error": str(e)})

print(f"\nPipeline complete. Processed {len(pipeline_results)} images.")

In [ ]:
# Build a summary DataFrame from the pipeline results
summary_records = []
for r in pipeline_results:
    catalog = r.get("catalog", {})
    summary_records.append({
        "image": r["image"],
        "status": r.get("status", "UNKNOWN"),
        "caption": r.get("caption", ""),
        "alt_text": r.get("alt_text", ""),
        "category": catalog.get("category", ""),
        "title": catalog.get("title", ""),
        "moderation_safe": r.get("moderation", {}).get("safe", None),
        "keywords": ", ".join(catalog.get("keywords", [])),
    })

summary_df = pd.DataFrame(summary_records)
print("Pipeline Summary:")
summary_df

In [ ]:
# Visualize category distribution from the pipeline
cat_counts = summary_df["category"].value_counts()

if not cat_counts.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    cat_counts.plot(kind="barh", ax=ax)
    ax.set_xlabel("Count")
    ax.set_title("Category Distribution — Pipeline Output")
    plt.tight_layout()
    plt.show()

In [ ]:
# Export the pipeline results to JSON for downstream use
output_path = "pipeline_results.json"
with open(output_path, "w") as f:
    json.dump(pipeline_results, f, indent=2, default=str)

print(f"Results saved to {output_path}")

# Summary

In this segment we covered:

1. **Image and Video Understanding** — Gemini's native multimodal architecture supports JPEG, PNG, WEBP, MP4, MOV, and more. Images can be passed as PIL objects, bytes, or base64.

2. **Image Captioning and VQA** — Prompt style determines output quality. Short, medium, and detailed captions serve different use cases. VQA enables interactive Q&A over image content.

3. **Multimodal Integration** — Multiple images, charts, and text can be combined in a single request. The Vertex AI `Part` API provides a production-grade way to assemble multimodal payloads.

4. **Real-world Pipelines** — Content moderation, alt text generation, document OCR, and visual search are all achievable with Gemini and well-crafted prompts. Combining them into a pipeline creates robust, automated workflows.

### Next Steps
- Explore `gemini-1.5-pro` for higher-quality outputs on complex documents
- Add function calling to ground responses in external databases
- Deploy pipelines to Cloud Run or Vertex AI Pipelines for production scale

<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>